# 🌦️ Rain Prediction — Model Training Walkthrough

This notebook mirrors and expands the ML pipeline used in the Weather App production code (`forecast/views.py`).  
It covers the complete workflow from raw CSV → trained model → saved artifact.

**Dataset:** `weather.csv` (368 records)  
**Target:** `RainTomorrow` (Yes / No)  
**Algorithm:** Random Forest Classifier  
**Library:** scikit-learn 1.7+

## 0 · Imports & Setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    mean_squared_error,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
)
import joblib

# Resolve the CSV path whether the notebook is run from the repo root or the notebooks/ folder
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
CSV_PATH = os.path.join(NOTEBOOK_DIR, '..', 'weather.csv')
CSV_PATH = os.path.normpath(CSV_PATH)

MODEL_OUTPUT_DIR = os.path.join(NOTEBOOK_DIR, 'saved_models')
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

print('CSV path:', CSV_PATH)
print('Model output dir:', MODEL_OUTPUT_DIR)

# Plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
FIGSIZE = (10, 5)

## 1 · Load & Inspect the Dataset

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

### 1.1 · Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

## 2 · Exploratory Data Analysis (EDA)

### 2.1 · Target Class Distribution

In [ ]:
counts = df['RainTomorrow'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

# Bar chart
axes[0].bar(counts.index, counts.values, color=['#4fc3f7', '#ef9a9a'], edgecolor='white', width=0.5)
axes[0].set_title('RainTomorrow — Count')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#4fc3f7', '#ef9a9a'], startangle=90, wedgeprops=dict(edgecolor='white'))
axes[1].set_title('RainTomorrow — Share')

plt.suptitle('Target Variable Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, 'class_distribution.png'), dpi=150)
plt.show()

### 2.2 · Numeric Feature Distributions

In [ ]:
numeric_features = ['MinTemp', 'MaxTemp', 'WindGustSpeed', 'Humidity', 'Pressure', 'Temp']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, col in zip(axes, numeric_features):
    for label, color in [('Yes', '#ef9a9a'), ('No', '#4fc3f7')]:
        subset = df[df['RainTomorrow'] == label][col].dropna()
        ax.hist(subset, bins=20, alpha=0.6, color=color, label=label, edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend(title='RainTomorrow')

plt.suptitle('Feature Distributions by Rain Label', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, 'feature_distributions.png'), dpi=150)
plt.show()

### 2.3 · Wind Gust Direction Frequency

In [ ]:
dir_counts = df['WindGustDir'].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(dir_counts.index, dir_counts.values, color=sns.color_palette('muted', len(dir_counts)))
ax.set_xlabel('Count')
ax.set_title('Wind Gust Direction — Frequency', fontweight='bold')
for bar in bars:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            str(int(bar.get_width())), va='center')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, 'wind_direction_frequency.png'), dpi=150)
plt.show()

### 2.4 · Correlation Heatmap

In [ ]:
# Encode for correlation
df_enc = df.copy()
le_tmp = LabelEncoder()
df_enc['WindGustDir'] = le_tmp.fit_transform(df_enc['WindGustDir'].astype(str))
df_enc['RainTomorrow'] = le_tmp.fit_transform(df_enc['RainTomorrow'].astype(str))

corr = df_enc.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, linewidths=0.5, ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, 'correlation_heatmap.png'), dpi=150)
plt.show()

## 3 · Data Preprocessing

In [ ]:
# ----- Exactly mirrors the production code in views.py -----

data = df.copy()

# Step 1: drop missing values and duplicates
before = len(data)
data = data.dropna().drop_duplicates()
after = len(data)
print(f'Rows removed: {before - after}  ({before} → {after})')

# Step 2: encode categorical columns
le_dir = LabelEncoder()          # for WindGustDir — reused at prediction time
le_rain = LabelEncoder()         # for RainTomorrow

data['WindGustDir']   = le_dir.fit_transform(data['WindGustDir'])
data['RainTomorrow']  = le_rain.fit_transform(data['RainTomorrow'])

print('\nWindGustDir classes:', list(le_dir.classes_))
print('RainTomorrow mapping:', dict(zip(le_rain.classes_, le_rain.transform(le_rain.classes_))))

# Step 3: define features (X) and target (y)
FEATURE_COLS = ['MinTemp', 'MaxTemp', 'WindGustDir', 'WindGustSpeed', 'Humidity', 'Pressure', 'Temp']
TARGET_COL   = 'RainTomorrow'

X = data[FEATURE_COLS]
y = data[TARGET_COL]

print(f'\nFeature matrix : {X.shape}')
print(f'Target vector  : {y.shape}')
print(f'\nClass balance (encoded):\n{y.value_counts()}')

## 4 · Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')
print(f'\nTrain class distribution:\n{y_train.value_counts()}')
print(f'\nTest  class distribution:\n{y_test.value_counts()}')

## 5 · Model Training — Random Forest Classifier

In [ ]:
# ----- Same hyperparameters as production code -----
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

print('Model trained successfully.')
print(f'Number of estimators : {rf_model.n_estimators}')
print(f'Max features         : {rf_model.max_features}')
print(f'Max depth            : {rf_model.max_depth} (None = unlimited)')

## 6 · Evaluation

In [ ]:
y_pred  = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
mse      = mean_squared_error(y_test, y_pred)
roc_auc  = roc_auc_score(y_test, y_proba)

print('=' * 40)
print(f'  Accuracy   : {accuracy:.4f}  ({accuracy*100:.2f}%)')
print(f'  MSE        : {mse:.4f}')
print(f'  ROC-AUC    : {roc_auc:.4f}')
print('=' * 40)

In [ ]:
print('Classification Report:')
print(classification_report(
    y_test, y_pred,
    target_names=le_rain.classes_
))

### 6.1 · Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred),
    display_labels=le_rain.classes_
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

### 6.2 · ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='#4fc3f7', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='#bdbdbd', lw=1, linestyle='--', label='Random baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Random Forest', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, 'roc_curve.png'), dpi=150)
plt.show()

### 6.3 · Cross-Validation

In [ ]:
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='accuracy')

print(f'5-Fold CV Accuracy scores : {[round(s, 4) for s in cv_scores]}')
print(f'Mean  : {cv_scores.mean():.4f}')
print(f'Std   : {cv_scores.std():.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, 6), cv_scores, color='#4fc3f7', edgecolor='white')
ax.axhline(cv_scores.mean(), color='#ef9a9a', linestyle='--', label=f'Mean = {cv_scores.mean():.4f}')
ax.set_xlabel('Fold')
ax.set_ylabel('Accuracy')
ax.set_title('5-Fold Cross-Validation Accuracy', fontweight='bold')
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, 'cross_validation.png'), dpi=150)
plt.show()

## 7 · Feature Importance

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(importances.index, importances.values,
               color=sns.color_palette('muted', len(importances)))
ax.set_xlabel('Importance (mean decrease in impurity)')
ax.set_title('Random Forest — Feature Importance', fontweight='bold')
for bar in bars:
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{bar.get_width():.4f}', va='center')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_OUTPUT_DIR, 'feature_importance.png'), dpi=150)
plt.show()

print('\nTop features driving rain prediction:')
print(importances.sort_values(ascending=False).to_string())

## 8 · Hyperparameter Tuning (GridSearchCV)

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth'   : [None, 10, 20],
    'min_samples_split': [2, 5],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

print('\nBest parameters :', grid_search.best_params_)
print('Best CV accuracy:', round(grid_search.best_score_, 4))

In [ ]:
# Evaluate the best model on the hold-out test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print('Best model — Test set accuracy :', round(accuracy_score(y_test, y_pred_best), 4))
print()
print(classification_report(y_test, y_pred_best, target_names=le_rain.classes_))

## 9 · Save the Trained Model

In [ ]:
MODEL_PATH   = os.path.join(MODEL_OUTPUT_DIR, 'rain_model.joblib')
ENCODER_PATH = os.path.join(MODEL_OUTPUT_DIR, 'label_encoder_wind.joblib')

joblib.dump(best_model, MODEL_PATH)
joblib.dump(le_dir,     ENCODER_PATH)

print(f'Model   saved → {MODEL_PATH}')
print(f'Encoder saved → {ENCODER_PATH}')

## 10 · Reload & Verify

In [ ]:
loaded_model   = joblib.load(MODEL_PATH)
loaded_encoder = joblib.load(ENCODER_PATH)

# Simulate a live prediction (same logic as views.py)
sample = {
    'MinTemp'     : 13.0,
    'MaxTemp'     : 25.0,
    'WindGustDir' : 'NW',      # compass direction from the API
    'WindGustSpeed': 45.0,
    'Humidity'    : 65.0,
    'Pressure'    : 1012.0,
    'Temp'        : 20.0,
}

# Encode wind direction
if sample['WindGustDir'] in loaded_encoder.classes_:
    sample['WindGustDir'] = loaded_encoder.transform([sample['WindGustDir']])[0]
else:
    sample['WindGustDir'] = -1  # unknown direction fallback

sample_df = pd.DataFrame([sample])
prediction = loaded_model.predict(sample_df)[0]
proba      = loaded_model.predict_proba(sample_df)[0]

outcome_map = {0: 'No Rain', 1: 'Rain'}
print(f'Prediction  : {outcome_map[prediction]}')
print(f'Probability : No Rain = {proba[0]:.2%} | Rain = {proba[1]:.2%}')

## 11 · Summary

| Step | Detail |
|---|---|
| Dataset | `weather.csv` — 368 records, 8 columns |
| Preprocessing | Drop NaN & duplicates · LabelEncode `WindGustDir` & `RainTomorrow` |
| Features | `MinTemp`, `MaxTemp`, `WindGustDir`, `WindGustSpeed`, `Humidity`, `Pressure`, `Temp` |
| Train/Test split | 80 % / 20 %, stratified |
| Baseline model | `RandomForestClassifier(n_estimators=100, random_state=42)` |
| Tuned model | Best params from 5-fold `GridSearchCV` |
| Outputs | `saved_models/rain_model.joblib` · `saved_models/label_encoder_wind.joblib` |
| Plots saved | class distribution · feature distributions · correlation heatmap · confusion matrix · ROC curve · cross-validation · feature importance |

> The production app (`forecast/views.py`) re-trains the baseline model on every request. For a higher-performance setup, replace the training step with `joblib.load('rain_model.joblib')` using the artifact saved here.